# CROPPER — recorte para OCR

O CROPPER **não treina**. Treino do detector: `OBB/train_yolov8_obb.ipynb`.

Fluxo: OBB → perspectiva 63×88 mm → inset 2% → CLAHE/nitidez → faixas de nome e rodapé.
O snap de borda na foto está **desligado** (corta nome / deixa vizinha).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

REPO = Path(".").resolve()
if not (REPO / "CROPPER" / "rectify.py").exists() and (REPO.parent / "CROPPER" / "rectify.py").exists():
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "CROPPER"))
from crop_from_obb import DEFAULT_IMGSZ, DEFAULT_WEIGHTS, run
from rectify import DEFAULT_DPI, card_size_px, crop_result

WEIGHTS = DEFAULT_WEIGHTS
OUT_DIR = REPO / "CROPPER" / "output" / "cards"
CONF = 0.8
IMGSZ = DEFAULT_IMGSZ
DPI = DEFAULT_DPI
REFINE = False
INSET = 0.02
ENHANCE = True
MAX_SHOW = 8

STEMS = ["IMG_6674", "IMG_6723", "IMG_7117", "IMG_6654", "IMG_6709"]

def find_images(stems: list[str] | None) -> list[Path]:
    dataset = REPO / "OBB" / "dataset"
    if not stems:
        return sorted((dataset / "test" / "images").glob("*.jpg"))
    found = []
    for stem in stems:
        hits = list(dataset.glob(f"*/images/*{stem}*"))
        if not hits:
            print(f"não achei {stem}")
            continue
        found.append(hits[0])
    return found

SOURCES = find_images(STEMS)
print(f"Pesos : {WEIGHTS}  exists={WEIGHTS.exists()}")
print(f"Saída : {OUT_DIR}  {card_size_px(DPI)[0]}×{card_size_px(DPI)[1]} px")
print(f"refine={REFINE}  inset={INSET}  enhance={ENHANCE}  fotos={len(SOURCES)}")


## 1. Recortar


In [ ]:
if not WEIGHTS.exists():
    raise FileNotFoundError(f"Treine o OBB antes. Faltando: {WEIGHTS}")
if not SOURCES:
    raise FileNotFoundError("Nenhuma foto.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(str(WEIGHTS))
saved: list[Path] = []
for src in SOURCES:
    saved.extend(
        run(
            source=src,
            weights=WEIGHTS,
            out_dir=OUT_DIR,
            conf=CONF,
            imgsz=IMGSZ,
            dpi=DPI,
            refine=REFINE,
            inset=INSET,
            enhance=ENHANCE,
            model=model,
        )
    )
print(f"{len(saved)} recorte(s) em {OUT_DIR}")


## 2. Cartas + faixas de OCR (nome e rodapé)


In [ ]:
from enhance import ocr_bands

previews = saved[:MAX_SHOW]
if not previews:
    print("Nenhuma carta recortada.")
else:
    fig, axes = plt.subplots(len(previews), 3, figsize=(10, 3.6 * len(previews)))
    if len(previews) == 1:
        axes = np.array([axes])
    for ax_row, path in zip(axes, previews):
        bgr = cv2.imread(str(path))
        bands = ocr_bands(bgr)
        ax_row[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        ax_row[0].set_title(path.name[:40], fontsize=8)
        ax_row[1].imshow(cv2.cvtColor(bands["name"], cv2.COLOR_BGR2RGB))
        ax_row[1].set_title("nome", fontsize=8)
        ax_row[2].imshow(cv2.cvtColor(bands["footer"], cv2.COLOR_BGR2RGB))
        ax_row[2].set_title("rodapé (set / nº)", fontsize=8)
        for ax in ax_row:
            ax.axis("off")
    plt.tight_layout()
    plt.show()
